In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split ,GridSearchCV ,cross_val_score
from sklearn.svm import SVC
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score ,confusion_matrix ,accuracy_score ,f1_score,classification_report ,mean_squared_error

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import BaggingRegressor ,RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.utils import class_weight

import numpy as np
import tensorflow as tf



/home/malak/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
2026-01-27 04:37:31.174588: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-27 04:37:31.193104: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-27 04:37:31.886560: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operation

In [2]:
df =pd.read_csv("cremad_features.csv")
df

,0,1,2,3,4,5,6,7,8,9,...,154,155,156,157,158,159,160,161,label,file_path
0,-306.027405,92.670242,8.491313,23.965405,7.477992,-5.759457,-11.883088,-9.676737,-3.996747,-13.352565,...,0.003351,0.002825,0.002283,0.002875,0.003217,0.002537,0.101868,1584.993071,angry,../AudioWAV/1001_DFA_ANG_XX.wav
1,-346.399628,95.839127,10.516282,31.619215,15.872088,-6.845448,-6.629935,-4.978728,-5.310655,-10.283518,...,0.001306,0.001039,0.001349,0.001003,0.001274,0.001167,0.093061,1531.650486,disgust,../AudioWAV/1001_DFA_DIS_XX.wav
2,-321.420258,94.760918,8.155398,23.323244,11.719157,-7.116333,-8.534803,-4.996965,-4.994401,-13.706510,...,0.017361,0.012410,0.019017,0.013379,0.010257,0.008362,0.084286,1489.088839,fear,../AudioWAV/1001_DFA_FEA_XX.wav
3,-303.303772,92.528893,4.231231,27.970137,10.869824,-11.878345,-10.095113,-7.149731,-7.651760,-17.085903,...,0.010875,0.005356,0.006022,0.003931,0.003002,0.003396,0.084878,1555.376035,happy,../AudioWAV/1001_DFA_HAP_XX.wav
4,-335.495972,100.393318,9.384934,30.160906,11.466775,-3.333670,-8.350987,-9.757346,-6.079329,-12.109532,...,0.000796,0.000431,0.000584,0.000571,0.000778,0.000657,0.082031,1495.394998,neutral,../AudioWAV/1001_DFA_NEU_XX.wav
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7437,-413.367371,96.386276,46.369545,14.591497,20.041862,10.173372,0.748282,-0.852761,4.764163,-0.944673,...,0.002370,0.002534,0.001539,0.000983,0.001509,0.001905,0.118683,1757.254778,disgust,../AudioWAV/1091_WSI_DIS_XX.wav
7438,-426.014099,91.230385,49.762840,20.573893,22.044111,11.756289,-1.450961,2.497097,5.650768,-3.032017,...,0.004076,0.004454,0.005628,0.004208,0.004655,0.007032,0.096364,1678.540253,fear,../AudioWAV/1091_WSI_FEA_XX.wav
7439,-370.487915,90.638107,38.969704,19.762012,14.836708,0.329151,-1.175138,-3.071633,5.731899,-3.763147,...,0.005281,0.006372,0.004301,0.003732,0.006234,0.006943,0.138205,1851.247161,happy,../AudioWAV/1091_WSI_HAP_XX.wav
7440,-393.181274,94.353287,45.251869,12.303626,12.717791,8.813284,0.926235,-3.176066,6.253240,-2.163018,...,0.002288,0.002203,0.002061,0.001348,0.001125,0.001000,0.113154,1788.313113,neutral,../AudioWAV/1091_WSI_NEU_XX.wav


In [3]:
df["label"] = df["label"].map({
    "angry": 0,
    "disgust": 1,
    "fear": 2,
    "sad": 3,
    "neutral": 4,
    "happy": 5,
})


In [4]:
df["label"].value_counts()

label
0    1271
1    1271
2    1271
5    1271
3    1271
4    1087
Name: count, dtype: int64

In [5]:
x =df.drop(["label" ,"file_path"] ,axis =1)
y =df["label"]

In [6]:
x_train , x_test ,y_train ,y_test =train_test_split(x ,y ,test_size=0.2 ,random_state=42 ,shuffle=True ,stratify=y)

In [7]:
scaler_x = StandardScaler()
x_train = scaler_x.fit_transform(x_train)
x_test = scaler_x.transform(x_test)

In [8]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)
x_train = pca.fit_transform(x_train)
x_test  = pca.transform(x_test)


In [9]:
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))


In [10]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=400,
    max_depth=5,
    min_samples_leaf=3,   
    min_samples_split=6,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42و
     class_weight={
        0: 1.0,
        1: 1.0,
        2: 1.3,
        3: 1.2,
        4: 1.3,
        5: 1.0
    }
)

model.fit(x_train, y_train)


y_pred_train = model.predict(x_train)
y_pred_test  = model.predict(x_test)

from sklearn.metrics import accuracy_score

train_acc = accuracy_score(y_train, y_pred_train)
test_acc  = accuracy_score(y_test, y_pred_test)

print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))

from sklearn.metrics import f1_score
print(f1_score(y_test, y_pred_test, average="macro"))





SyntaxError: invalid decimal literal (2053659619.py, line 10)

In [ ]:
from sklearn.svm import SVC

model = SVC(
    kernel="rbf",
    C=1,
    gamma=0.1,
    class_weight={
        0: 1.0,
        1: 1.0,
        2: 1.3,
        3: 1.2,
        4: 1.3,
        5: 1.0
    }
)


model.fit(x_train, y_train)


y_pred_train = model.predict(x_train)
y_pred_test  = model.predict(x_test)

from sklearn.metrics import accuracy_score

train_acc = accuracy_score(y_train, y_pred_train)
test_acc  = accuracy_score(y_test, y_pred_test)

print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))



from sklearn.metrics import f1_score
print(f1_score(y_test, y_pred_test, average="macro"))




Train accuracy: 0.9333109356626911
Test accuracy: 0.4613834788448623
[[198  12   4   1   5  34]
 [ 47  79  20  35  39  34]
 [ 73  14  78  44  28  17]
 [  8  23  31 154  29  10]
 [ 13  35  13  32  96  29]
 [ 94  23  17  11  27  82]]
              precision    recall  f1-score   support

           0       0.46      0.78      0.58       254
           1       0.42      0.31      0.36       254
           2       0.48      0.31      0.37       254
           3       0.56      0.60      0.58       255
           4       0.43      0.44      0.43       218
           5       0.40      0.32      0.36       254

    accuracy                           0.46      1489
   macro avg       0.46      0.46      0.45      1489
weighted avg       0.46      0.46      0.45      1489

0.4465781817186916
